In [57]:
from dotenv import load_dotenv
import os

from typing import TypedDict


from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

In [58]:
load_dotenv()

OPEN_AI_API = os.getenv("OPENAI_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")

print(os.getenv("LANGSMITH_TRACING"))
print(os.getenv("LANGSMITH_PROJECT"))
print(bool(os.getenv("LANGSMITH_API_KEY")))

true
support-assistant
True


In [53]:
from typing import Annotated
import operator
from langchain_core.messages import HumanMessage, AIMessage

class State(TypedDict):
    messages: Annotated[list, add_messages]
    notes: Annotated[list, operator.add]

def normalize(state: State):
    tmp = state['messages'][-1].content
    tmp = tmp.lower().strip()
    
    return {"messages": [HumanMessage(content=tmp)]}

def respond(state: State):

    llm = ChatOpenAI(model="gpt-4")
    response = llm.invoke(state['messages'])
    return {"messages": [AIMessage(content=response.content)]}


builder = StateGraph(State)

builder.add_node("normalize", normalize)
builder.add_node("respond", respond)

builder.add_edge(START, "normalize")
builder.add_edge("normalize", "respond")
builder.add_edge("respond", END)


graph = builder.compile()

result = graph.invoke({"messages": ["Explain how LLM works?"]})

new = result['messages'] + [HumanMessage(content="Now explain it like I'm five.")]
result2 = graph.invoke({"messages": new})

In [ ]:
from typing import Annotated
import operator
from langchain_core.messages import HumanMessage, AIMessage

class InputState(TypedDict):
    question: str

class OutputState(TypedDict):
    answer: str


class State(TypedDict):
    question: str
    answer: str
    messages: Annotated[list, add_messages]
    notes: Annotated[list, operator.add]

def normalize(state: State):
    tmp = state['question']
    tmp = tmp.lower().strip()
    
    return {"messages": [HumanMessage(content=tmp)]}

def respond(state: State):

    llm = ChatOpenAI(model="gpt-4")
    response = llm.invoke(state['messages'])
    return {"messages": [AIMessage(content=response.content)], "answer":state['answer']}


builder = StateGraph(State, input_schema=InputState, output_schema=OutputState)

builder.add_node("normalize", normalize)
builder.add_node("respond", respond)

builder.add_edge(START, "normalize")
builder.add_edge("normalize", "respond")
builder.add_edge("respond", END)


graph = builder.compile()

result = graph.invoke({"question": ["Explain how LLM works?"]})

new = result['messages'] + [HumanMessage(content="Now explain it like I'm five.")]
result2 = graph.invoke({"question": new})

3
6


In [64]:
from typing import Annotated
import operator
from langchain_core.messages import HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from typing import TypedDict


class InputState(TypedDict):
    question: str


class OutputState(TypedDict):
    answer: str


class State(TypedDict):
    question: str
    answer: str
    messages: Annotated[list, add_messages]
    notes: Annotated[list, operator.add]


def line():
    print("-" * 70)


def normalize(state: State):
    print()
    print("=" * 70)
    print("[STEP 2] normalize() -- INPUT")
    print("=" * 70)

    print("type(state):", type(state))
    print()
    print("state contents:")
    for k, v in state.items():
        print(f"    {k!r:12} -> {v!r}")
    line()

    tmp = state['question']
    print(f"raw question        : {tmp!r}")

    tmp = tmp.lower().strip()
    print(f"after lower().strip(): {tmp!r}")
    line()

    output = {"messages": [HumanMessage(content=tmp)]}

    print()
    print("=" * 70)
    print("[STEP 3] normalize() -- OUTPUT (partial update sent back to graph)")
    print("=" * 70)
    print("output dict:")
    for k, v in output.items():
        print(f"    {k!r:12} -> {v!r}")
    print()

    return output


def respond(state: State):
    print()
    print("=" * 70)
    print("[STEP 4] respond() -- INPUT (state after normalize's update merged in)")
    print("=" * 70)

    print("state contents:")
    for k, v in state.items():
        print(f"    {k!r:12} -> {v!r}")
    line()

    last_msg = state['messages'][-1]
    print("last message object :", last_msg)
    print("type               :", type(last_msg))
    print()

    llm = ChatOpenAI(model="gpt-4")
    response = llm.invoke(state['messages'])

    print("=" * 70)
    print("[STEP 5] respond() -- raw LLM call result")
    print("=" * 70)
    print("type(response)     :", type(response))
    print("response.content   :", repr(response.content))
    print()

    output = {
        "messages": [AIMessage(content=response.content)],
        "answer": response.content,
    }

    print("=" * 70)
    print("[STEP 6] respond() -- OUTPUT (partial update sent back to graph)")
    print("=" * 70)
    print("output keys:", list(output.keys()))
    print()

    return output


builder = StateGraph(State, input_schema=InputState, output_schema=OutputState)

builder.add_node("normalize", normalize)
builder.add_node("respond", respond)

builder.add_edge(START, "normalize")
builder.add_edge("normalize", "respond")
builder.add_edge("respond", END)

graph = builder.compile()

print()
print("#" * 70)
print("[STEP 1] BEFORE invoke -- raw dict passed by caller (no validation here)")
print("#" * 70)

input_payload = {"question": "Explain how LLM works?"}
print("type(input_payload):", type(input_payload))
print("input_payload      :", input_payload)

result = graph.invoke(input_payload)

print()
print("#" * 70)
print("[STEP 7] AFTER invoke -- final state, filtered down to OutputState")
print("#" * 70)
print("type(result):", type(result))
print("result      :", result)
print()



######################################################################
[STEP 1] BEFORE invoke -- raw dict passed by caller (no validation here)
######################################################################
type(input_payload): <class 'dict'>
input_payload      : {'question': 'Explain how LLM works?'}

[STEP 2] normalize() -- INPUT
type(state): <class 'dict'>

state contents:
    'question'   -> 'Explain how LLM works?'
    'messages'   -> []
    'notes'      -> []
----------------------------------------------------------------------
raw question        : 'Explain how LLM works?'
after lower().strip(): 'explain how llm works?'
----------------------------------------------------------------------

[STEP 3] normalize() -- OUTPUT (partial update sent back to graph)
output dict:
    'messages'   -> [HumanMessage(content='explain how llm works?', additional_kwargs={}, response_metadata={})]


[STEP 4] respond() -- INPUT (state after normalize's update merged in)
state contents:
 

In [50]:
new

[HumanMessage(content='Explain how LLM works?', additional_kwargs={}, response_metadata={}, id='05147381-b29d-41f5-b113-0b1e1553f8f7'),
 HumanMessage(content='explain how llm works?', additional_kwargs={}, response_metadata={}, id='4c05cfc7-5be7-48fe-b3fc-3a0ef4475241'),
 AIMessage(content='LLM stands for Log-linear Models. These are statistical models commonly used in various fields such as machine learning and natural language processing. \n\nLLM generally works by using mathematical functions, known as log-linear functions, to predict the likelihood of a certain outcomes or events based on input data. These models aim to predict the probability distribution of a certain outcome given a set of input features. \n\nHere\'s a simple process what LLM might entail: \n\n1. Define the features: To start, features related to the outcomes you\'re seeking are identified. For instance, in natural language processing, the features might include elements like the frequency of certain words or phr